# Chapitre 3 — Données & Prétraitement
## Polish Companies Bankruptcy Data — Année 5

**TER — Modèle Stochastique de Merton vs Machine Learning — Prédiction du Risque de Crédit**
ISFA — Master 1 Actuariat — 2025/2026

---
**Source :** UCI ML Repository — https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data

**Objectif :** Charger les données brutes, explorer les distributions, sélectionner les variables
pertinentes pour Merton et les modèles ML, puis construire un pipeline de prétraitement rigoureux.

---
## 0. Imports

In [ ]:
import pandas as pd            # Manipulation de données tabulaires
import numpy as np             # Calcul numérique vectorisé
import matplotlib.pyplot as plt  # Visualisation
import matplotlib.patches as mpatches
import seaborn as sns          # Heatmaps et statistiques visuelles
import warnings
warnings.filterwarnings('ignore')

from scipy.io import arff      # Lecture du format ARFF (UCI ML Repository)
from scipy import stats

# Pipeline sklearn : split, imputation, normalisation
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# SMOTE : sur-échantillonnage synthétique pour corriger le déséquilibre de classes
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print("SMOTE disponible.")
except ImportError:
    SMOTE_AVAILABLE = False
    print("SMOTE non dispo — pip install imbalanced-learn")

# Palette de couleurs  sur tous les graphiques du TER
BLUE   = '#2C5F8A'   # Classe 0 = entreprises solvables (non-défaut)
PURPLE = '#7B3FA0'   # Classe 1 = entreprises défaillantes (défaut)
GRAY   = '#9E9E9E'   # Neutre / modèle de référence


---
## 3.1 — Chargement des données

In [ ]:
# ── Lecture du fichier .arff ──────────────────────────────────────────────────
# Le format ARFF est propre au dépôt UCI. scipy.io.arff.loadarff() retourne
# un tableau numpy structuré (comme un CSV mais typé) + les métadonnées.
raw, meta = arff.loadarff('5year.arff')
df = pd.DataFrame(raw)

# ── Décodage de la variable cible ─────────────────────────────────────────────
# Dans ARFF, les chaînes sont des bytes : b'0', b'1'.
# On convertit en int pour sklearn.
# class = 0 -> l'entreprise n'a PAS fait faillite dans les 5 ans suivants
# class = 1 -> l'entreprise a fait faillite dans les 5 ans suivants
df['class'] = df['class'].apply(lambda x: int(x.decode('utf-8')))

# ── Renommage Attr1..Attr64 -> A1..A64 (convention plus concise) ──────────────
df.rename(columns={f'Attr{i}': f'A{i}' for i in range(1, 65)}, inplace=True)

print(f"Dataset Year 5 : {df.shape[0]:,} observations x {df.shape[1]} colonnes")
print()

# ── Distribution des classes ──────────────────────────────────────────────────
# Le déséquilibre (~97% / ~3%) est typique du risque de crédit :
# les défauts sont des événements rares. Ce point justifie :
#   (1) l'utilisation de métriques adaptées (AUC-ROC, F1 plutôt qu'accuracy)
#   (2) le recours au SMOTE pour rééquilibrer le train set
vc = df['class'].value_counts().sort_index()
print("Distribution des classes :")
print(f"  Non-défaut (0) : {vc[0]:,}  ({vc[0]/len(df)*100:.2f}%)")
print(f"  Défaut     (1) : {vc[1]:,}  ({vc[1]/len(df)*100:.3f}%)")
print(f"  Ratio : 1 défaut pour {int(vc[0]/vc[1])} non-défauts")


---
## 3.2 — Sélection des variables

In [ ]:
# ── Sélection de 12 ratios financiers parmi les 64 disponibles ───────────────
# Critères de sélection (double validation) :
#   (1) Pertinence économique : ces ratios couvrent les grandeurs implicitement
#       utilisées par le modèle de Merton (rentabilité des actifs = dérive mu_V,
#       levier financier = distance au seuil de défaut, liquidité = coussin)
#   (2) Couverture des 5 dimensions du risque de crédit classiquement retenues :
#       profitabilité, solvabilité, liquidité, efficience, taille relative

SELECTED = ['A1','A2','A3','A4','A7','A10','A16','A17','A18','A29','A36','A40']

# Description détaillée de chaque variable et son lien avec Merton
VAR_DEF = {
    'A1' : ('Profit net / Total actif',
            'Profitabilite — proxy de la derive mu_V des actifs dans Merton'),
    'A2' : ('Passif total / Total actif',
            'Levier financier — correspond au ratio Debt/Value de Merton'),
    'A3' : ('Résultat courant / Total actif',
            'Profitabilite courante — ratio de l Altman Z-score'),
    'A4' : ('Tresorerie / Total actif',
            'Liquidite court terme — coussin face aux obligations de dette'),
    'A7' : ('EBIT / Total actif',
            'Rentabilite economique — Return on Assets avant interet et impots'),
    'A10': ('Capitaux propres / Total actif',
            'Ratio fonds propres — inverse du levier, cle dans Merton'),
    'A16': ('Passif courant / Total actif',
            'Poids des dettes CT — risque de liquidite immediat'),
    'A17': ('Actif courant / Passif courant',
            'Ratio de liquidite courante — capacite a honorer les dettes CT'),
    'A18': ('Profit brut / Total actif',
            'Marge brute rapportee aux actifs — rentabilite operationnelle'),
    'A29': ('Capitaux propres / Passif total',
            'Structure de financement — lien direct avec le seuil de defaut'),
    'A36': ('Charges financieres / Total actif',
            'Cout de la dette — pression du service de la dette'),
    'A40': ('Fonds de roulement / Total actif',
            'Liquidite nette — solde actif circulant hors dettes CT'),
}

print(f"{'Var':<6} {'Formule':<42} Intuition Merton")
print("-"*90)
for var, (formule, intuition) in VAR_DEF.items():
    print(f"{var:<6} {formule:<42} {intuition}")

# Extraction de la matrice features et du vecteur cible
X_all = df[SELECTED].copy()  # (n x 12) : les 12 ratios financiers
y     = df['class'].copy()   # (n,) : 0 = sain, 1 = défaut


---
## 3.3 — Statistiques descriptives

In [ ]:
# ── Audit des valeurs manquantes ──────────────────────────────────────────────
# Les ratios financiers ont souvent des NaN (division par zéro, non-reporté).
# On quantifie le problème AVANT de choisir la stratégie d'imputation.
miss = X_all.isnull().sum().sort_values(ascending=False)
total_miss  = miss.sum()
total_cells = X_all.shape[0] * X_all.shape[1]

print(f"Valeurs manquantes : {total_miss:,} / {total_cells:,}  ({total_miss/total_cells*100:.2f}%)")
print()
print("Par variable (variables avec au moins un NaN) :")
for var, n in miss[miss > 0].items():
    print(f"  {var} : {n:,}  ({n/len(df)*100:.1f}%)")
# Conclusion : taux modéré -> imputation médiane suffit.
# La médiane est préférée à la moyenne car les distributions sont asymétriques
# (queues épaisses dues aux entreprises extrêmes).


In [ ]:
# ── Fonctions de winsorisation ────────────────────────────────────────────────
# Les ratios financiers présentent des outliers extrêmes (faillite imminente).
# On applique une winsorisation P1-P99 : valeurs < P1 -> P1, > P99 -> P99.
# Cela élimine les valeurs aberrantes SANS perdre les observations.
#
# RÈGLE CLÉ : les bornes sont calculées UNIQUEMENT sur le train set.
# Appliquer les bornes du test pour calculer celles du train = DATA LEAKAGE.

def fit_winsorise(df_train, lo=0.01, hi=0.99):
    '''
    Calcule les bornes de winsorisation [P_lo, P_hi] sur le train set.
    Retourne un dict {colonne: (borne_basse, borne_haute)}.
    lo=0.01 -> on coupe 1% des valeurs les plus basses et les plus hautes.
    '''
    return {c: (df_train[c].quantile(lo), df_train[c].quantile(hi))
            for c in df_train.columns}

def apply_winsorise(df_in, bounds):
    '''
    Applique les bornes pré-calculées (depuis le train) a n'importe quel set.
    df.clip(lo, hi) : toute valeur < lo -> lo, > hi -> hi (operateur de winsorisation).
    On passe TOUJOURS les bornes du train, meme pour le test.
    '''
    out = df_in.copy()
    for c, (lo, hi) in bounds.items():
        if c in out.columns:
            out[c] = out[c].clip(lower=lo, upper=hi)
    return out

# Winsorisation provisoire sur X_all pour les statistiques descriptives
# (les vraies bornes train-only seront calculées après le split)
from sklearn.model_selection import train_test_split as _spl
_Xtr, _, _, _ = _spl(X_all, y, test_size=0.2, random_state=42, stratify=y)
_bounds  = fit_winsorise(_Xtr)
X_w      = apply_winsorise(X_all, _bounds)   # version winsorisée (pour stats)
X_nbk_w  = X_w[y == 0]                       # sous-ensemble solvables
X_bk_w   = X_w[y == 1]                       # sous-ensemble défaillants

print("Statistiques descriptives — variables winsorisées P1-P99 :")
desc = X_w.describe(percentiles=[.25,.5,.75]).round(4)
print(desc.T[['mean','std','25%','50%','75%']].to_string())


In [ ]:
# ── Histogrammes superposés : non-défaut vs défaut ────────────────────────────
# Objectif : visualiser le pouvoir discriminant individuel de chaque variable.
# Si les deux distributions sont bien séparées -> la variable est un bon prédicteur.
# Si elles se superposent totalement -> la variable discrimine peu seule.
# Avec density=True on normalise l'aire = 1 pour comparer deux groupes de
# tailles très différentes (97% vs 3%).

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, var in enumerate(SELECTED):
    ax  = axes[i]
    d0  = X_nbk_w[var].dropna()
    d1  = X_bk_w[var].dropna()
    lo  = min(d0.quantile(.01), d1.quantile(.01))
    hi  = max(d0.quantile(.99), d1.quantile(.99))
    bins = np.linspace(lo, hi, 30)

    ax.hist(d0, bins=bins, color=BLUE,   alpha=0.5, density=True,
            edgecolor='white', lw=0.4, label='Non-défaut (0)')
    ax.hist(d1, bins=bins, color=PURPLE, alpha=0.5, density=True,
            edgecolor='white', lw=0.4, label='Défaut (1)')

    ax.set_title(var, fontsize=9, fontweight='bold', color='#333')
    ax.set_ylabel('Densité', fontsize=7)
    ax.tick_params(labelsize=7)
    ax.spines[['top','right']].set_visible(False)
    if i == 0:
        ax.legend(fontsize=7)

plt.suptitle('Distributions des 12 ratios par statut (winsorisés P1-P99)',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_distributions.pdf', bbox_inches='tight', dpi=150)
plt.show()
# Lecture : A2 (levier) et A10 (fonds propres/actif) montrent une bonne séparation
# -> bons prédicteurs du risque, cohérent avec la logique de Merton.


In [ ]:
# ── Matrice de corrélation de Pearson ─────────────────────────────────────────
# Objectif : détecter la multicolinéarité entre les 12 variables.
# Forte corrélation (|rho| > 0.7) -> information redondante -> problème pour la
# régression logistique (gonflement de la variance des coefficients).
# XGBoost (basé sur des arbres) est beaucoup moins sensible a ce problème.
# La régularisation L1 de la LR permettra de gérer les variables corrélées.

corr = X_w.corr().round(2)

fig, ax = plt.subplots(figsize=(11, 9))
# Diverging colormap centré sur 0 : bleu = corrélation négative, violet = positive
cmap = sns.diverging_palette(220, 280, s=80, l=40, as_cmap=True)
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', annot_kws={'size': 7.5},
            cmap=cmap, center=0, vmin=-1, vmax=1,
            linewidths=0.4, linecolor='white',
            square=True, cbar_kws={'shrink': 0.8})

ax.set_title('Matrice de corrélation de Pearson — 12 variables financières\n'
             '(winsorisées P1-P99)', fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig_correlation.pdf', bbox_inches='tight', dpi=150)
plt.show()
# A2 et A10 sont fortement corrélées (~-0.9) : dette/actif vs CP/actif sont
# complémentaires par construction (l'un est quasi l'inverse de l'autre).


---
## 3.4 — Pipeline de prétraitement

In [ ]:
# ════════════════════════════════════════════════════════════════════
# RÈGLE FONDAMENTALE : PAS DE DATA LEAKAGE
# ════════════════════════════════════════════════════════════════════
# Tout paramètre appris (bornes de winsorisation, médianes d'imputation,
# mean/std pour standardisation) DOIT être calculé UNIQUEMENT sur le
# train set, puis appliqué tel quel au test set.
# Sinon le modèle "voit" indirectement des informations futures -> biais.
# ════════════════════════════════════════════════════════════════════

# ── Étape 1 : Découpe stratifiée 80 / 20 ─────────────────────────────────────
# stratify=y : impose que la proportion de défauts (~3%) soit identique
#              dans le train ET dans le test -> évaluation fiable
# random_state=42 : seed fixe pour reproductibilité complète

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train : {len(X_train):,} obs  |  Défauts : {y_train.sum()}  ({y_train.mean()*100:.2f}%)")
print(f"Test  : {len(X_test):,}  obs  |  Défauts : {y_test.sum()}   ({y_test.mean()*100:.2f}%)")
print("-> Proportions quasi-identiques : stratification validée")


In [ ]:
# ── Étape 2 : Winsorisation (bornes sur le train uniquement) ──────────────────
# On recalcule maintenant les bornes proprement sur X_train.
# Les mêmes bornes sont ensuite appliquées a X_test :
# on simule ce qui se passerait en déploiement réel (on ne connait pas le test).

win_bounds = fit_winsorise(X_train, lo=0.01, hi=0.99)
X_train_w  = apply_winsorise(X_train, win_bounds)
X_test_w   = apply_winsorise(X_test,  win_bounds)   # <- bornes du train

print("Exemples de bornes de winsorisation (calculées sur le train) :")
for var in ['A1', 'A2', 'A10', 'A29']:
    lo, hi = win_bounds[var]
    print(f"  {var} : P1 = {lo:+.4f}  |  P99 = {hi:+.4f}")


In [ ]:
# ── Étape 3 : Imputation des NaN par la médiane ───────────────────────────────
# Pourquoi la médiane ?
#   - Les distributions financières sont asymétriques (queues épaisses).
#   - La médiane est robuste aux outliers résiduels après winsorisation.
#   - La moyenne serait tirée par les valeurs extrêmes.
# fit_transform sur X_train : on APPREND les médianes sur le train uniquement.
# transform sur X_test      : on APPLIQUE les médianes du train au test.

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train_w),     # apprend + transforme
    columns=SELECTED, index=X_train_w.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test_w),          # transforme avec les médianes du train
    columns=SELECTED, index=X_test_w.index
)

print("Médianes d'imputation (apprises sur le train) :")
for var, med in zip(SELECTED, imputer.statistics_):
    print(f"  {var} : {med:+.5f}")

print(f"\nNaN restants - Train : {X_train_imp.isnull().sum().sum()}  |  Test : {X_test_imp.isnull().sum().sum()}")


In [ ]:
# ── Étape 4 : Standardisation (centrage-réduction) ────────────────────────────
# Pourquoi standardiser pour la régression logistique ?
#   - La LR minimise sa fonction de coût par descente de gradient.
#   - Si les variables ont des échelles très différentes (A4 dans [0,1],
#     A36 dans [-50,50]), le gradient est mal conditionné -> convergence lente.
#   - Après standardisation : chaque variable a mean~0 et std~1.
# XGBoost N'a PAS besoin de standardisation :
#   - Les arbres de décision font des comparaisons de seuils -> scale-invariant.
#   - On utilisera donc X_train_imp / X_test_imp directement pour XGBoost.

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_imp),    # apprend mean et std sur le train
    columns=SELECTED, index=X_train_imp.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_imp),         # applique les mêmes mean/std au test
    columns=SELECTED, index=X_test_imp.index
)

print("Vérification après standardisation (doit être ~0 et ~1 sur le train) :")
print(f"  Train — Mean : {X_train_scaled.mean().mean():.6f}  |  Std : {X_train_scaled.std().mean():.6f}")
print(f"  Test  — Mean : {X_test_scaled.mean().mean():.4f}  |  Std : {X_test_scaled.std().mean():.4f}")
print("  (Le test n'est pas exactement centré-réduit car on utilise les params du train)")


In [ ]:
# ── Étape 5 : SMOTE — Synthetic Minority Over-sampling Technique ──────────────
# Problème : avec 97% / 3% de déséquilibre, un classifieur naif qui prédit
# TOUJOURS "non-défaut" atteint 97% d'accuracy... mais 0% de rappel sur les défauts.
#
# SMOTE génère des observations SYNTHÉTIQUES de la classe minoritaire (défauts) :
#   Pour chaque observation défaillante :
#   1. On identifie ses k plus proches voisins dans la classe minoritaire
#   2. On crée un point synthétique par interpolation linéaire entre
#      l'observation et l'un de ses voisins (position aléatoire sur le segment)
#
# RÈGLE ABSOLUE : SMOTE s'applique UNIQUEMENT au train set.
#   - Le test doit refléter la distribution réelle (~3% de défauts)
#     pour évaluer les performances en conditions réelles.
#   - Appliquer SMOTE au test gonflerait artificiellement les métriques.

if SMOTE_AVAILABLE:
    smote = SMOTE(random_state=42)  # k_neighbors=5 par défaut
    X_train_sm, y_train_sm = smote.fit_resample(X_train_imp, y_train)
    X_train_sm = pd.DataFrame(X_train_sm, columns=SELECTED)

    vc_b = pd.Series(y_train).value_counts().sort_index()
    vc_a = pd.Series(y_train_sm).value_counts().sort_index()
    print(f"Avant SMOTE : Classe 0 = {vc_b[0]:,}  |  Classe 1 = {vc_b[1]:,}  (ratio {vc_b[0]//vc_b[1]}:1)")
    print(f"Apres SMOTE : Classe 0 = {vc_a[0]:,}  |  Classe 1 = {vc_a[1]:,}  (ratio {vc_a[0]//vc_a[1]}:1)")
    print(f"Obs. synthétiques ajoutées : {vc_a[1] - vc_b[1]:,}")
    print("Le test set reste INCHANGÉ (distribution réelle)")
else:
    X_train_sm, y_train_sm = X_train_imp, y_train
    print("SMOTE non dispo — données non rééchantillonnées.")


In [ ]:
# ── Standardisation des données SMOTE (pour la Régression Logistique) ─────────
# On re-standardise apres SMOTE : les points synthétiques peuvent légèrement
# décaler la distribution. On refit le scaler sur le train SMOTE complet.

if SMOTE_AVAILABLE:
    scaler_sm = StandardScaler()
    X_train_sm_sc = pd.DataFrame(
        scaler_sm.fit_transform(X_train_sm),   # fit sur train SMOTE
        columns=SELECTED
    )
    X_test_sm_sc = pd.DataFrame(
        scaler_sm.transform(X_test_imp),        # transform du test avec params SMOTE
        columns=SELECTED, index=X_test_imp.index
    )
    print("Données prêtes pour la Régression Logistique (standardisées + SMOTE).")
else:
    X_train_sm_sc, X_test_sm_sc = X_train_scaled, X_test_scaled
    print("Données standardisées sans SMOTE.")


---
## Récapitulatif du pipeline

In [ ]:
# ── Tableau récapitulatif des 7 étapes du pipeline ────────────────────────────
pipeline = pd.DataFrame({
    'Etape': [
        '1. Chargement 5year.arff',
        '2. Sélection des variables',
        '3. Découpe stratifiée 80/20',
        '4. Winsorisation P1-P99',
        '5. Imputation médiane',
        '6. Standardisation',
        '7. SMOTE (train uniquement)'
    ],
    'Detail': [
        f'{len(df):,} obs x 64 variables — cible : class (0/1)',
        '12 ratios financiers (profitabilité, levier, liquidité)',
        f'Train : {len(X_train):,} | Test : {len(X_test):,} — stratifié',
        'Bornes calculées sur le train, appliquées au test',
        'Médianes calculées sur le train, appliquées au test',
        'Centrage-réduction (Régression Logistique uniquement)',
        f'Train : {len(X_train_sm):,} obs équilibrées | Test inchangé : {len(X_test_imp):,}' if SMOTE_AVAILABLE else 'Non appliqué'
    ],
    'Usage': [
        'df brut',
        'X_all, y',
        'X_train, X_test, y_train, y_test',
        'X_train_w, X_test_w',
        'X_train_imp, X_test_imp  <-- pour XGBoost',
        'X_train_scaled, X_test_scaled',
        'X_train_sm_sc, X_test_sm_sc  <-- pour Régr. Logistique'
    ]
})
print(pipeline.to_string(index=False))
